# 02 – Mô hình truyền thống: LDA (A1), LSA + K-Means (A2)
**Đầu ra:** `results.csv`, `topic_words.csv`, cột `lda` và `lsa_kmeans` của `doc_topics.csv`, `models/lda/`

In [2]:
import sys; sys.path.append("..")
from src.config import *

In [3]:
import time
import pandas as pd
from src.preprocess import load_articles, get_tokens
from src.metrics import evaluate, make_row, save_results
from src.baselines import run_lda, run_lsa_kmeans, save_lda

df = load_articles()
texts = get_tokens(df)              # list[list[str]]: cho LDA và C_v
text_tok = df["text_tok"].tolist()  # list[str]: cho LSA
y = df["true_label"].values         # nhãn thật, chỉ dùng để chấm điểm
print("Số bài:", len(df))

Số bài: 1800


In [ ]:
rows = []
for seed in SEEDS:
    t0 = time.perf_counter()
    labels, topic_words, _, _ = run_lda(texts, seed=seed)
    time_s = time.perf_counter() - t0
    scores = evaluate(labels, y, topic_words, texts)
    rows.append(make_row("lda", "bow", "k6", "", seed, scores, time_s))
    print(f"LDA seed={seed}: NMI={scores['nmi']:.3f}  C_v={scores['cv']:.3f}  ({time_s:.1f}s)")

for n_dims in SVD_DIMS:
    for seed in SEEDS:
        t0 = time.perf_counter()
        labels, topic_words, _ = run_lsa_kmeans(text_tok, n_dims, seed=seed)
        time_s = time.perf_counter() - t0
        scores = evaluate(labels, y, topic_words, texts)
        rows.append(make_row("lsa_kmeans", "tfidf_svd", "k6", f"svd_dim={n_dims}", seed, scores, time_s))
        print(f"LSA {n_dims} chiều, seed={seed}: NMI={scores['nmi']:.3f}  C_v={scores['cv']:.3f}")

save_results(rows)

LDA seed=42: NMI=0.509  C_v=0.707  (22.8s)
LDA seed=43: NMI=0.478  C_v=0.635  (25.2s)
LDA seed=44: NMI=0.522  C_v=0.695  (22.8s)
LSA 10 chiều, seed=42: NMI=0.615  C_v=0.739
LSA 10 chiều, seed=43: NMI=0.669  C_v=0.793
LSA 10 chiều, seed=44: NMI=0.629  C_v=0.734
LSA 20 chiều, seed=42: NMI=0.676  C_v=0.781
LSA 20 chiều, seed=43: NMI=0.680  C_v=0.775
LSA 20 chiều, seed=44: NMI=0.672  C_v=0.782
LSA 50 chiều, seed=42: NMI=0.642  C_v=0.793
LSA 50 chiều, seed=43: NMI=0.643  C_v=0.795
LSA 50 chiều, seed=44: NMI=0.668  C_v=0.788


In [ ]:
res = pd.read_csv(RESULTS_CSV, keep_default_na=False)
res = res[res["method"].isin(["lda", "lsa_kmeans"])]
cols = ["ari", "nmi", "purity", "cv", "diversity", "time_s"]
summary = res.groupby(["method", "param"])[cols].agg(["mean", "std"]).round(3)
summary

ari           nmi        purity            cv         \
                        mean    std   mean    std   mean    std   mean    std   
method     param                                                                
lda                    0.448  0.032  0.503  0.022  0.652  0.021  0.679  0.039   
lsa_kmeans svd_dim=10  0.592  0.027  0.638  0.028  0.769  0.041  0.755  0.033   
           svd_dim=20  0.627  0.004  0.676  0.004  0.822  0.002  0.780  0.004   
           svd_dim=50  0.572  0.046  0.651  0.015  0.786  0.027  0.792  0.004   

                      diversity         time_s         
                           mean    std    mean    std  
method     param                                       
lda                       0.922  0.025  23.636  1.384  
lsa_kmeans svd_dim=10     0.967  0.000   1.871  0.240  
           svd_dim=20     0.967  0.000   1.779  0.173  
           svd_dim=50     0.978  0.010   2.031  0.049

In [ ]:
cv_by_dim = res[res["method"] == "lsa_kmeans"].groupby("param")["cv"].mean()
print(cv_by_dim.round(3))
best_dim = int(cv_by_dim.idxmax().split("=")[1])
print("Số chiều SVD được chọn (C_v cao nhất):", best_dim)

param
svd_dim=10    0.755
svd_dim=20    0.780
svd_dim=50    0.792
Name: cv, dtype: float64
Số chiều SVD được chọn (C_v cao nhất): 50


In [ ]:
lda_labels, lda_words, lda, dictionary = run_lda(texts, seed=SEED)
lsa_labels, lsa_words, _ = run_lsa_kmeans(text_tok, best_dim, seed=SEED)
save_lda(lda, dictionary)

def topic_rows(method, labels, words):
    counts = pd.Series(labels).value_counts()
    return [{"method": method, "topic": t, "n_docs": int(counts.get(t, 0)), "words": " ".join(w)}
            for t, w in enumerate(words)]

tw = pd.DataFrame(topic_rows("lda", lda_labels, lda_words) + topic_rows("lsa_kmeans", lsa_labels, lsa_words))
if TOPIC_WORDS_CSV.exists():
    old = pd.read_csv(TOPIC_WORDS_CSV)
    tw = pd.concat([old[~old["method"].isin(["lda", "lsa_kmeans"])], tw], ignore_index=True)
tw.to_csv(TOPIC_WORDS_CSV, index=False, encoding="utf-8")
pd.set_option("display.max_colwidth", None)
tw

,method,topic,n_docs,words
0,lda,0,278,việt_nam đội trận bóng đội_tuyển giải cầu_thủ hlv thi_đấu thắng
1,lda,1,159,nvidia công_ty huang ceo mỹ lãnh_đạo vụ jensen chip công_nghệ
2,lda,2,281,đồng giá tỷ triệu giảm usd tăng tiền thị_trường đầu_tư
3,lda,3,398,bác_sĩ bệnh dùng điều_trị người_bệnh bệnh_viện giảm tăng tình_trạng y_tế
4,lda,4,349,việt_nam doanh_nghiệp công_nghệ nước kinh_tế đầu_tư xây_dựng sản_xuất sản_phẩm nghiên_cứu
5,lda,5,335,phim diễn_viên chương_trình văn_hóa khán_giả tuổi gia_đình ăn bộ việt_nam
6,lsa_kmeans,0,272,trận đội đội_tuyển bóng cầu_thủ hlv việt_nam u17 giải thắng
7,lsa_kmeans,1,198,nvidia usd đồng huang giá cổ_phiếu tỷ công_ty ceo thị_trường
8,lsa_kmeans,2,288,bác_sĩ điều_trị người_bệnh bệnh bệnh_viện ung_thư ăn thuốc đau thận
9,lsa_kmeans,3,589,doanh_nghiệp việt_nam công_nghệ dự_án đầu_tư đất dân kinh_tế tỉnh nước


In [ ]:
dt = pd.read_csv(DOC_TOPICS_CSV) if DOC_TOPICS_CSV.exists() else df[["id", "true_label"]].copy()
dt["lda"] = lda_labels
dt["lsa_kmeans"] = lsa_labels
dt.to_csv(DOC_TOPICS_CSV, index=False, encoding="utf-8")

In [ ]:
for name, lab in [("LDA", lda_labels), (f"LSA ({best_dim} chiều)", lsa_labels)]:
    print(name)
    display(pd.crosstab(pd.Series(lab, name="Chủ đề"), df["true_label"].map(TOPICS).rename("Nhãn thật")))

LDA


Nhãn thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Chủ đề,,,,,,
0,2,0,1,0,267,8
1,54,7,32,1,0,65
2,10,4,143,5,8,111
3,98,0,10,268,6,16
4,128,7,108,9,8,89
5,8,282,6,17,11,11


LSA (50 chiều)


Nhãn thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Chủ đề,,,,,,
0,2,1,0,1,268,0
1,48,6,137,0,6,1
2,0,1,0,270,6,11
3,87,29,150,23,15,285
4,1,263,6,5,5,2
5,162,0,7,1,0,1


In [ ]:
from sklearn.metrics import silhouette_score

sil_rows = []
for n_dims in SVD_DIMS:
    for seed in SEEDS:
        labels, _, X_lsa = run_lsa_kmeans(text_tok, n_dims, seed=seed)
        sil_rows.append({"svd_dim": n_dims, "seed": seed, "silhouette": silhouette_score(X_lsa, labels)})
sil = pd.DataFrame(sil_rows)
sil.groupby("svd_dim")["silhouette"].agg(["mean", "std"]).round(3)

,mean,std
svd_dim,,
10,0.391,0.004
20,0.231,0.001
50,0.128,0.001


In [ ]:
def vn(x, d=3):
    return f"{x:.{d}f}".replace(".", ",")   # 0.448 -> 0,448

def ms(s, d=3):
    return f"{vn(s.mean(), d)} ± {vn(s.std(), d)}"   # trung bình ± độ lệch chuẩn

print("### Mục 1. Tóm tắt (trung bình ± độ lệch chuẩn qua 3 seed)\n")
print("| Phương pháp | Cấu hình chọn | ARI | NMI | Purity | C_v | Diversity | Thời gian (s) |")
print("|---|---|---|---|---|---|---|---|")
for name, method, param, cfg in [("A1. LDA (k = 6)", "lda", "", f"BoW, passes = {LDA_PASSES}"),
                                 ("A2. LSA + K-Means (k = 6)", "lsa_kmeans", f"svd_dim={best_dim}", f"TF-IDF → SVD {best_dim} chiều")]:
    g = res[(res["method"] == method) & (res["param"] == param)]
    print(f"| {name} | {cfg} | " + " | ".join(ms(g[c]) for c in ["ari", "nmi", "purity", "cv", "diversity"])
          + f" | {vn(g['time_s'].mean(), 1)} |")

print("\n### Mục 3. Chọn siêu tham số (trung bình 3 seed)\n")
print("| Số chiều SVD | C_v | Silhouette | NMI (tham khảo) |")
print("|---|---|---|---|")
for d in SVD_DIMS:
    g = res[res["param"] == f"svd_dim={d}"]
    s = sil.loc[sil["svd_dim"] == d, "silhouette"]
    mark = " ✔" if d == best_dim else ""
    print(f"| {d}{mark} | {ms(g['cv'])} | {ms(s)} | {vn(g['nmi'].mean())} |")

print("\n### Mục 4. Chủ đề tìm được (seed 42)\n")
print("| Chủ đề | LDA | LSA + K-Means |")
print("|---|---|---|")
lda_n, lsa_n = pd.Series(lda_labels).value_counts(), pd.Series(lsa_labels).value_counts()
for t in range(N_TOPICS):
    print(f"| {t} | {', '.join(lda_words[t])} ({lda_n.get(t, 0)} bài) | {', '.join(lsa_words[t])} ({lsa_n.get(t, 0)} bài) |")

### Mục 1. Tóm tắt (trung bình ± độ lệch chuẩn qua 3 seed)

| Phương pháp | Cấu hình chọn | ARI | NMI | Purity | C_v | Diversity | Thời gian (s) |
|---|---|---|---|---|---|---|---|
| A1. LDA (k = 6) | BoW, passes = 10 | 0,448 ± 0,032 | 0,503 ± 0,022 | 0,652 ± 0,021 | 0,679 ± 0,039 | 0,922 ± 0,025 | 23,6 |
| A2. LSA + K-Means (k = 6) | TF-IDF → SVD 50 chiều | 0,572 ± 0,046 | 0,651 ± 0,015 | 0,786 ± 0,027 | 0,792 ± 0,004 | 0,978 ± 0,010 | 2,0 |

### Mục 3. Chọn siêu tham số (trung bình 3 seed)

| Số chiều SVD | C_v | Silhouette | NMI (tham khảo) |
|---|---|---|---|
| 10 | 0,755 ± 0,033 | 0,391 ± 0,004 | 0,638 |
| 20 | 0,780 ± 0,004 | 0,231 ± 0,001 | 0,676 |
| 50 ✔ | 0,792 ± 0,004 | 0,128 ± 0,001 | 0,651 |

### Mục 4. Chủ đề tìm được (seed 42)

| Chủ đề | LDA | LSA + K-Means |
|---|---|---|
| 0 | việt_nam, đội, trận, bóng, đội_tuyển, giải, cầu_thủ, hlv, thi_đấu, thắng (278 bài) | trận, đội, đội_tuyển, bóng, cầu_thủ, hlv, việt_nam, u17, giải, thắng (272 bài) |
| 1 | nvidia, công_ty, huang,

## Phân tích lỗi LDA và BERTopic

In [4]:
dt = pd.read_csv(DOC_TOPICS_CSV)
d = dt.merge(df[["id", "title"]], on="id")
MODELS = ["lda", "lsa_kmeans", "bertopic_k6"]
for m in MODELS:
    ct = pd.crosstab(d[m], d["true_label"])
    d[m + "_nhan"] = d[m].map(ct.idxmax(axis=1))      # nhãn đa số của chủ đề
    d[m + "_dung"] = d[m + "_nhan"] == d["true_label"]
print(d[[m + "_dung" for m in MODELS]].mean().round(3))
pd.crosstab(d["lda_dung"], d["bertopic_k6_dung"], margins=True)

lda_dung            0.641
lsa_kmeans_dung     0.769
bertopic_k6_dung    0.808
dtype: float64


bertopic_k6_dung,False,True,All
lda_dung,,,
False,210,437,647
True,136,1017,1153
All,346,1454,1800


In [5]:
for m in ["lda", "bertopic_k6"]:
    sai = d[~d[m + "_dung"]]
    cap = (sai.groupby(["true_label", m + "_nhan"]).size().sort_values(ascending=False).head(5)
             .rename("so_bai").reset_index())
    cap["true_label"] = cap["true_label"].map(TOPICS)
    cap[m + "_nhan"] = cap[m + "_nhan"].map(TOPICS)
    print(f"\n{m}: {len(sai)} bài bị xếp sai")
    print(cap.to_string(index=False))


lda: 647 bài bị xếp sai
true_label  lda_nhan  so_bai
   Thời sự   Kinh tế     111
   Kinh tế Công nghệ     108
 Công nghệ  Sức khỏe      98
   Thời sự Công nghệ      89
 Công nghệ   Thời sự      54

bertopic_k6: 346 bài bị xếp sai
true_label bertopic_k6_nhan  so_bai
   Thời sự          Kinh tế      64
 Công nghệ          Kinh tế      57
   Kinh tế        Công nghệ      45
   Kinh tế          Thời sự      43
   Thời sự         Sức khỏe      18


In [6]:
pd.set_option("display.max_colwidth", 90)
cols = ["id", "title", "true_label", "lda", "lda_nhan", "bertopic_k6", "bertopic_k6_nhan"]
lda_sai = d[~d["lda_dung"] & d["bertopic_k6_dung"]]
bert_sai = d[d["lda_dung"] & ~d["bertopic_k6_dung"]]
print("LDA sai, BERTopic đúng:", len(lda_sai))
display(lda_sai.sample(10, random_state=SEED)[cols])
print("LDA đúng, BERTopic sai:", len(bert_sai))
display(bert_sai.sample(10, random_state=SEED)[cols])

LDA sai, BERTopic đúng: 437


,id,title,true_label,lda,lda_nhan,bertopic_k6,bertopic_k6_nhan
1651,doc_1652,Chủ tịch Trung Quốc kêu gọi gỡ bỏ rào cản thương mại,kinh_te,4,cong_nghe,0,kinh_te
379,doc_0380,"Ra mắt Liên minh Kinh tế tầm thấp: Khai phá ""mỏ vàng"" 10 tỷ USD",kinh_te,4,cong_nghe,0,kinh_te
1217,doc_1218,Xem bộ đội luyện bắn súng đại liên PKMS trên xe thiết giáp chở quân BTR-152,thoi_su,3,suc_khoe,2,thoi_su
1330,doc_1331,Thứ trưởng Đào Xuân Cơ: Việt Nam có cơ hội giới thiệu thế mạnh về y tế,suc_khoe,4,cong_nghe,1,suc_khoe
1408,doc_1409,Nỗi lo khi xe khách chuyển về bến xe Yên Sở,thoi_su,2,kinh_te,2,thoi_su
1258,doc_1259,Dell Việt Nam cảnh báo từ chối bảo hành với laptop 'hàng nhập khẩu',cong_nghe,3,suc_khoe,5,cong_nghe
957,doc_0958,"Tội phạm mạng dùng AI 'thay áo' kịch bản cũ, hack Zalo người dùng",cong_nghe,3,suc_khoe,5,cong_nghe
390,doc_0391,Ngày Nhà giáo đặc biệt của 100 thầy cô ở viện ung thư,suc_khoe,5,giai_tri,1,suc_khoe
1780,doc_1781,"Microsoft xin lỗi sau sự cố dịch vụ tại 74 quốc gia và vùng lãnh thổ, công bố nguyên nhân",cong_nghe,3,suc_khoe,5,cong_nghe
1681,doc_1682,VNPT và Qualcomm thử nghiệm thiết bị Wi-Fi 8 tích hợp AI,cong_nghe,3,suc_khoe,5,cong_nghe


LDA đúng, BERTopic sai: 136


,id,title,true_label,lda,lda_nhan,bertopic_k6,bertopic_k6_nhan
818,doc_0819,"Đề xuất giảm phạt hơn 12.600 tỷ đồng cho người nộp thuế trạng thái 03, 06",kinh_te,2,kinh_te,2,thoi_su
604,doc_0605,'Human' là Từ của năm 2026,giai_tri,5,giai_tri,5,cong_nghe
709,doc_0710,"Việt Nam - Hàn Quốc ký 8 thỏa thuận hợp tác AI, bán dẫn",cong_nghe,4,cong_nghe,0,kinh_te
557,doc_0558,Thí nghiệm dò hạt ma lớn nhất thế giới,cong_nghe,4,cong_nghe,0,kinh_te
1704,doc_1705,Việt Nam đẩy mạnh đưa AI vào hoạt động thực tế,cong_nghe,4,cong_nghe,0,kinh_te
724,doc_0725,"""Cội nguồn của hạnh phúc"" - Cuốn sách giải thích hành vi của con người",giai_tri,5,giai_tri,1,suc_khoe
171,doc_0172,"Cấm chiêu tạo khan hiếm giả, ép người dùng mua hàng online",kinh_te,2,kinh_te,5,cong_nghe
896,doc_0897,Giám đốc bị bắn ở Đồng Nai nguy kịch,suc_khoe,3,suc_khoe,2,thoi_su
272,doc_0273,Giải Caine bác cáo buộc 'nhà văn dùng AI',giai_tri,5,giai_tri,5,cong_nghe
771,doc_0772,Dương Yến Ngọc mắc ung thư,giai_tri,5,giai_tri,1,suc_khoe


In [7]:
def xem(doc_id):
    r = d.set_index("id").loc[doc_id]
    tok = df.set_index("id").loc[doc_id, "text_tok"].split()
    print(doc_id, "|", r["title"])
    print("  Nhãn thật:", TOPICS[r["true_label"]], "| LDA:", r["lda"], "→", TOPICS[r["lda_nhan"]],
          "| BERTopic:", r["bertopic_k6"], "→", TOPICS[r["bertopic_k6_nhan"]])
    print("  Từ xuất hiện nhiều nhất:", pd.Series(tok).value_counts().head(8).index.tolist())

for doc_id in ["doc_0958", "doc_0886", "doc_0746", "doc_0206", "doc_0140", "doc_1208", "doc_0697", "doc_0292"]:
    xem(doc_id)

doc_0958 | Tội phạm mạng dùng AI 'thay áo' kịch bản cũ, hack Zalo người dùng
  Nhãn thật: Công nghệ | LDA: 3 → Sức khỏe | BERTopic: 5 → Công nghệ
  Từ xuất hiện nhiều nhất: ['tài_khoản', 'mạng', 'dùng', 'zalo', 'tiền', 'chuyển', 'xã_hội', 'tội_phạm']
doc_0886 | Jensen Huang từ chức CEO Nvidia sau bê bối tình cảm
  Nhãn thật: Công nghệ | LDA: 1 → Thời sự | BERTopic: 5 → Công nghệ
  Từ xuất hiện nhiều nhất: ['nvidia', 'ceo', 'jensen_huang', 'công_ty', 'huang', 'từ_chức', 'mỹ', 'quan_hệ']
doc_0746 | Bị phạt từ 5 triệu đồng trở xuống có thể thay bằng lao động công ích
  Nhãn thật: Thời sự | LDA: 2 → Kinh tế | BERTopic: 2 → Thời sự
  Từ xuất hiện nhiều nhất: ['phạt', 'tiền', 'chính_phủ', 'quốc_hội', 'biện_pháp', 'xử_phạt', 'đồng', 'đề_nghị']
doc_0206 | Ăn khoai lang lúc nào tốt nhất? Điều nhiều người có thể chưa biết
  Nhãn thật: Sức khỏe | LDA: 5 → Giải trí | BERTopic: 1 → Sức khỏe
  Từ xuất hiện nhiều nhất: ['ăn', 'khoai_lang', 'bữa', 'chất', 'xơ', 'khẩu_phần', 'thực_phẩm', 'củ']
doc_0140